## Weather JSON Inspection

Inspect the actual JSON structure stored in the Bronze Weather table.

In [0]:
-- Check the Bronze Weather table structure and content
SELECT 
  source_system,
  source_url,
  source_file,
  batch_id,
  ingested_at,
  LENGTH(raw_json) AS json_length_bytes
FROM `ftw-week-08`.`01_bronze`.`weather_raw`

In [0]:
-- Inspect the JSON structure - look at the top-level keys and hourly data structure
WITH parsed AS (
  SELECT from_json(raw_json, 'latitude DOUBLE, longitude DOUBLE, timezone STRING, timezone_abbreviation STRING, hourly STRUCT<time: ARRAY<STRING>, temperature_2m: ARRAY<DOUBLE>, precipitation: ARRAY<DOUBLE>, rain: ARRAY<DOUBLE>, snowfall: ARRAY<DOUBLE>, weather_code: ARRAY<INT>, wind_speed_10m: ARRAY<DOUBLE>>') AS json_data
  FROM `ftw-week-08`.`01_bronze`.`weather_raw`
)
SELECT 
  json_data.latitude AS latitude,
  json_data.longitude AS longitude,
  json_data.timezone AS timezone,
  json_data.timezone_abbreviation AS timezone_abbreviation,
  SIZE(json_data.hourly.time) AS hourly_time_count,
  SIZE(json_data.hourly.temperature_2m) AS hourly_temperature_count,
  SIZE(json_data.hourly.precipitation) AS hourly_precipitation_count,
  SIZE(json_data.hourly.rain) AS hourly_rain_count,
  SIZE(json_data.hourly.snowfall) AS hourly_snowfall_count,
  SIZE(json_data.hourly.weather_code) AS hourly_weather_code_count,
  SIZE(json_data.hourly.wind_speed_10m) AS hourly_wind_speed_count
FROM parsed

In [0]:
-- Sample first and last few hourly timestamps from the JSON payload
WITH parsed AS (
  SELECT from_json(raw_json, 'latitude DOUBLE, longitude DOUBLE, timezone STRING, timezone_abbreviation STRING, hourly STRUCT<time: ARRAY<STRING>, temperature_2m: ARRAY<DOUBLE>, precipitation: ARRAY<DOUBLE>, rain: ARRAY<DOUBLE>, snowfall: ARRAY<DOUBLE>, weather_code: ARRAY<INT>, wind_speed_10m: ARRAY<DOUBLE>>') AS json_data
  FROM `ftw-week-08`.`01_bronze`.`weather_raw`
)
SELECT 
  json_data.hourly.time[0] AS first_timestamp,
  json_data.hourly.time[1] AS second_timestamp,
  json_data.hourly.time[2] AS third_timestamp,
  json_data.hourly.time[SIZE(json_data.hourly.time) - 3] AS third_to_last_timestamp,
  json_data.hourly.time[SIZE(json_data.hourly.time) - 2] AS second_to_last_timestamp,
  json_data.hourly.time[SIZE(json_data.hourly.time) - 1] AS last_timestamp
FROM parsed

## Weather Silver Transformation

Create the Silver Weather table by parsing and flattening the JSON hourly arrays into one row per hourly timestamp.

In [0]:
-- Create Silver Weather table by exploding and flattening the hourly JSON arrays
CREATE OR REPLACE TABLE `ftw-week-08`.`02_silver`.`weather` AS
WITH parsed_json AS (
  SELECT 
    from_json(raw_json, 'latitude DOUBLE, longitude DOUBLE, timezone STRING, timezone_abbreviation STRING, hourly STRUCT<time: ARRAY<STRING>, temperature_2m: ARRAY<DOUBLE>, precipitation: ARRAY<DOUBLE>, rain: ARRAY<DOUBLE>, snowfall: ARRAY<DOUBLE>, weather_code: ARRAY<INT>, wind_speed_10m: ARRAY<DOUBLE>>') AS json_data,
    source_system,
    source_url,
    source_file,
    batch_id,
    ingested_at
  FROM `ftw-week-08`.`01_bronze`.`weather_raw`
),
exploded_weather AS (
  SELECT 
    -- Explode the hourly time array with position to maintain alignment
    POSEXPLODE(json_data.hourly.time) AS (hour_index, hourly_time_str),
    json_data,
    source_system,
    source_url,
    source_file,
    batch_id,
    ingested_at
  FROM parsed_json
)
SELECT
  -- Convert hourly timestamp string to TIMESTAMP type
  CAST(hourly_time_str AS TIMESTAMP) AS weather_datetime,
  
  -- Extract weather measurements by position using hour_index to maintain alignment
  json_data.hourly.temperature_2m[hour_index] AS temperature_2m,
  json_data.hourly.precipitation[hour_index] AS precipitation,
  json_data.hourly.rain[hour_index] AS rain,
  json_data.hourly.snowfall[hour_index] AS snowfall,
  json_data.hourly.weather_code[hour_index] AS weather_code,
  json_data.hourly.wind_speed_10m[hour_index] AS wind_speed_10m,
  
  -- Bronze provenance columns
  source_system,
  source_url,
  source_file,
  batch_id,
  ingested_at
FROM exploded_weather
ORDER BY weather_datetime

## Weather Silver Validation

Validate the Silver Weather table structure and data quality.

In [0]:
-- Validate schema and data types
DESCRIBE `ftw-week-08`.`02_silver`.`weather`

In [0]:
-- Basic row count and timestamp range
SELECT
  COUNT(*) AS total_row_count,
  MIN(weather_datetime) AS min_weather_datetime,
  MAX(weather_datetime) AS max_weather_datetime,
  DATEDIFF(DAY, MIN(weather_datetime), MAX(weather_datetime)) + 1 AS days_covered
FROM `ftw-week-08`.`02_silver`.`weather`

In [0]:
-- Check for duplicate timestamps
SELECT
  weather_datetime,
  COUNT(*) AS occurrence_count
FROM `ftw-week-08`.`02_silver`.`weather`
GROUP BY weather_datetime
HAVING COUNT(*) > 1
ORDER BY occurrence_count DESC, weather_datetime

In [0]:
-- NULL count for every weather field
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN weather_datetime IS NULL THEN 1 ELSE 0 END) AS null_weather_datetime,
  SUM(CASE WHEN temperature_2m IS NULL THEN 1 ELSE 0 END) AS null_temperature_2m,
  SUM(CASE WHEN precipitation IS NULL THEN 1 ELSE 0 END) AS null_precipitation,
  SUM(CASE WHEN rain IS NULL THEN 1 ELSE 0 END) AS null_rain,
  SUM(CASE WHEN snowfall IS NULL THEN 1 ELSE 0 END) AS null_snowfall,
  SUM(CASE WHEN weather_code IS NULL THEN 1 ELSE 0 END) AS null_weather_code,
  SUM(CASE WHEN wind_speed_10m IS NULL THEN 1 ELSE 0 END) AS null_wind_speed_10m
FROM `ftw-week-08`.`02_silver`.`weather`

In [0]:
-- Check for timestamps outside the requested period (2026-03-01 through 2026-05-31)
SELECT
  COUNT(*) AS timestamps_outside_period,
  MIN(weather_datetime) AS earliest_outside,
  MAX(weather_datetime) AS latest_outside
FROM `ftw-week-08`.`02_silver`.`weather`
WHERE weather_datetime < '2026-03-01 00:00:00'
   OR weather_datetime > '2026-05-31 23:59:59'

## Weather Coverage and Data Quality

Detect missing hourly timestamps and compare actual vs. expected observations.

In [0]:
-- Detect missing hourly timestamps by comparing each timestamp with the next
WITH ordered_timestamps AS (
  SELECT 
    weather_datetime,
    LEAD(weather_datetime) OVER (ORDER BY weather_datetime) AS next_datetime,
    TIMESTAMPDIFF(HOUR, weather_datetime, LEAD(weather_datetime) OVER (ORDER BY weather_datetime)) AS hours_gap
  FROM `ftw-week-08`.`02_silver`.`weather`
)
SELECT
  weather_datetime AS timestamp_before_gap,
  next_datetime AS timestamp_after_gap,
  hours_gap,
  hours_gap - 1 AS missing_hours
FROM ordered_timestamps
WHERE hours_gap > 1
ORDER BY weather_datetime

In [0]:
-- Compare the number of timestamps in the JSON payload vs. Silver row count
WITH bronze_count AS (
  SELECT SIZE(from_json(raw_json, 'latitude DOUBLE, longitude DOUBLE, timezone STRING, timezone_abbreviation STRING, hourly STRUCT<time: ARRAY<STRING>, temperature_2m: ARRAY<DOUBLE>, precipitation: ARRAY<DOUBLE>, rain: ARRAY<DOUBLE>, snowfall: ARRAY<DOUBLE>, weather_code: ARRAY<INT>, wind_speed_10m: ARRAY<DOUBLE>>').hourly.time) AS json_hourly_count
  FROM `ftw-week-08`.`01_bronze`.`weather_raw`
),
silver_count AS (
  SELECT COUNT(*) AS silver_row_count
  FROM `ftw-week-08`.`02_silver`.`weather`
)
SELECT
  b.json_hourly_count,
  s.silver_row_count,
  b.json_hourly_count - s.silver_row_count AS difference
FROM bronze_count b, silver_count s

In [0]:
-- Expected vs actual observations summary
WITH stats AS (
  SELECT
    COUNT(*) AS actual_observations,
    MIN(weather_datetime) AS min_datetime,
    MAX(weather_datetime) AS max_datetime,
    TIMESTAMPDIFF(HOUR, MIN(weather_datetime), MAX(weather_datetime)) + 1 AS expected_hourly_observations
  FROM `ftw-week-08`.`02_silver`.`weather`
)
SELECT
  actual_observations,
  expected_hourly_observations,
  expected_hourly_observations - actual_observations AS missing_observations,
  ROUND(100.0 * actual_observations / expected_hourly_observations, 2) AS coverage_percentage,
  min_datetime,
  max_datetime
FROM stats

In [0]:
-- Sample first 10 weather records
SELECT *
FROM `ftw-week-08`.`02_silver`.`weather`
ORDER BY weather_datetime
LIMIT 10

In [0]:
-- Sample last 10 weather records
SELECT *
FROM `ftw-week-08`.`02_silver`.`weather`
ORDER BY weather_datetime DESC
LIMIT 10

### Validation Summary

**Silver Weather Table: `ftw-week-08`.`02_silver`.`weather`**

**Data Quality:**
* **Total Observations:** 2,208 hourly records
* **Expected Observations:** 2,208 (92 days × 24 hours)
* **Coverage:** 100.00% - Perfect hourly coverage with no gaps

**Timestamp Range:**
* **Start:** 2026-03-01 00:00:00 (America/New_York)
* **End:** 2026-05-31 23:00:00 (America/New_York)
* **Duration:** 92 days

**Data Integrity:**
* **Duplicate Timestamps:** 0
* **NULL Values:** 0 across all weather fields
* **Out-of-Range Timestamps:** 0
* **Missing Hourly Gaps:** 0
* **JSON-to-Silver Alignment:** Perfect match (2,208 = 2,208)

**Schema:**
* `weather_datetime` (TIMESTAMP)
* `temperature_2m` (DOUBLE)
* `precipitation` (DOUBLE)
* `rain` (DOUBLE)
* `snowfall` (DOUBLE)
* `weather_code` (INT)
* `wind_speed_10m` (DOUBLE)
* Bronze provenance: `source_system`, `source_url`, `source_file`, `batch_id`, `ingested_at`

**Status:** All validation checks passed. The Silver Weather table is complete and ready for downstream Gold transformations.